# Poisson correction for GLORYS12 (U, V, W)

Make the column-integrated horizontal transport non-divergent, then rebuild W by upward
integration from $w(-H)=0$. That gives $W(0)=0$ **and** exact 3-D non-divergence at the same
time, without deleting anything from W.

Per column, the depth-integrated divergence is

$$D = \frac{1}{A}\sum_k \Big[F^u_i - F^u_{i-1} + F^v_j - F^v_{j-1}\Big]_k ,$$

which is exactly the surface value you get by integrating the budget up from the seabed.
Solve for a barotropic potential $\phi$,

$$\nabla\!\cdot\!\big(H\,\nabla\phi\big) = D\,A ,\qquad
\delta u = -\frac{\partial\phi}{\partial x},\quad \delta v = -\frac{\partial\phi}{\partial y},$$

add $(\delta u, \delta v)$ at every level, and re-integrate. The correction is the smallest
depth-uniform one that removes $D$.

**Two things about this cut-out that the algorithm has to handle:**

1. The ocean here is **two disconnected basins** (49 465 and ~374 000 points) — a single pinned
   point leaves the second basin singular.
2. The domain rim is an **open** boundary. Treating it as a closed wall (pure Neumann) makes the
   problem incompatible: $\sum D A \neq 0$, because water genuinely enters and leaves the
   cut-out. `BC="open"` puts $\phi=0$ on the rim and lets the imbalance flow out where it
   physically goes; this is what reaches machine zero. `BC="closed"` follows the pinned-Neumann
   recipe and gets ~7e-8 instead of ~3e-18.

**Variant B — linearly weighted correction.** The correction is applied as
$g(z) = 2(z+H)/H$: zero at the seabed, double at the surface, column mean 1. The
depth-integrated transport correction is identical to the uniform version, so $W(0)=0$ still
holds exactly, but currents at 100-500 m are left almost untouched.


In [1]:
year, month = 1993, 1

ZGR    = "Zgr_cmesh2.nc"                     # mbathy, e3t_0, e3t_ps, gdepw_0
HGR    = "Hgr_cmesh.nc"                      # e1t, e2t, e1u, e2u, e1v, e2v
U_PATH = f"U_{year}-{month:02d}.nc"
V_PATH = f"V_{year}-{month:02d}.nc"
W_PATH = f"W_{year}-{month:02d}.nc"          # GLORYS, for validation only

U_OUT  = f"U_{year}-{month:02d}b.nc"
V_OUT  = f"V_{year}-{month:02d}b.nc"
W_OUT  = f"W_{year}-{month:02d}b.nc"

BASIN    = "largest"  # "largest": keep only the biggest connected body of water and treat the
                      #   rest as land. In this cut-out that drops the eastern Pacific, which
                      #   Central America separates from the Atlantic. "all": keep everything.
BC       = "open"     # "open": phi=0 on the domain rim.  "closed": Neumann + pin per basin.
COMPRESS = False

In [2]:
import numpy as np
import xarray as xr
from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.sparse.csgraph import connected_components

## 1. Grid

In [3]:
zgr = xr.open_dataset(ZGR).squeeze()
hgr = xr.open_dataset(HGR).squeeze()

mbathy  = zgr.mbathy.values.astype(int)
ny, nx  = mbathy.shape

if BASIN == "largest":
    # Label the connected bodies of water (two cells are connected if they share a face) and
    # keep only the biggest. Everything else becomes land, so U, V and W are written as 0 there
    # and no particle can enter.
    _oc  = mbathy > 0
    _idx = np.full((ny, nx), -1, np.int64); _idx[_oc] = np.arange(_oc.sum())
    _r = np.concatenate([_idx[:, :-1][_oc[:, :-1] & _oc[:, 1:]],
                         _idx[:-1, :][_oc[:-1, :] & _oc[1:, :]]])
    _c = np.concatenate([_idx[:, 1:][_oc[:, :-1] & _oc[:, 1:]],
                         _idx[1:, :][_oc[:-1, :] & _oc[1:, :]]])
    _n, _lab = connected_components(
        sparse.coo_matrix((np.ones(_r.size), (_r, _c)), shape=(_oc.sum(),) * 2), directed=False)
    _full = np.full((ny, nx), -1); _full[_oc] = _lab
    _dropped = _oc & (_full != np.bincount(_lab).argmax())
    mbathy = np.where(_dropped, 0, mbathy)
    print(f"{_n} separate bodies of water; keeping the largest, "
          f"masking {int(_dropped.sum())} cells as land")
nz      = zgr.sizes["z"]
gdepw_0 = zgr.gdepw_0.squeeze().values.astype("f8")
gdept_0 = zgr.gdept_0.squeeze().values.astype("f8")
kz      = np.arange(nz)[:, None, None]

# some NEMO builds name these e1t_0 etc.
def hv(name):
    for n in (name, name + "_0"):
        if n in hgr:
            return hgr[n].values.astype("f8")
    raise KeyError(name)

e1t, e2t = hv("e1t"), hv("e2t")
e1u, e2u = hv("e1u"), hv("e2u")
e1v, e2v = hv("e1v"), hv("e2v")
area = e1t * e2t

# T-cell thickness with partial steps
e3t = np.where(kz + 1 <= mbathy - 1, zgr.e3t_0.squeeze().values[:, None, None], zgr.e3t_ps.values)
e3t = np.where(kz + 1 <= mbathy, e3t, 0.0)

def _face_min(a, axis):
    b = np.minimum(a, np.roll(a, -1, axis=axis))
    if axis == 2: b[:, :, -1] = a[:, :, -1]
    else:         b[:, -1, :] = a[:, -1, :]
    return b

e3u = _face_min(e3t, 2)          # thickness seen by the u-face east of T(i)
e3v = _face_min(e3t, 1)          # thickness seen by the v-face north of T(j)
Hu, Hv = e3u.sum(0), e3v.sum(0)
H      = e3t.sum(0)

ocean = mbathy > 0
print(f"{ny} x {nx} x {nz},  {ocean.sum()} ocean columns,  deepest {H.max():.0f} m")

2 separate bodies of water; keeping the largest, masking 49984 cells as land
499 x 1260 x 50,  375377 ocean columns,  deepest 5958 m


## 2. The Laplacian, built once

In [4]:
Cu = e2u * Hu / e1u              # coupling through the u-face  (0 where the face is closed)
Cv = e1v * Hv / e2v

# Unknowns: ocean points strictly inside the rim. The rim itself has no closed budget
# (j=0 has no Fv(j-1), i=0 has no Fu(i-1)) and is where the open boundary lives.
unknown = ocean.copy()
unknown[0, :] = unknown[-1, :] = False
unknown[:, 0] = unknown[:, -1] = False
# with BC="closed" the rim is a wall: faces touching it are dropped entirely
neigh = ocean if BC == "open" else unknown

idx = np.full((ny, nx), -1, np.int64)
idx[unknown] = np.arange(unknown.sum())
n_unk = int(unknown.sum())

rows, cols, vals = [], [], []
diag = np.zeros(n_unk)

# east faces: T(j,i) - T(j,i+1), coefficient Cu(j,i)
# north faces: T(j,i) - T(j+1,i), coefficient Cv(j,i)
for a2, b2, coef, face in (
        (idx[:, :-1], idx[:, 1:], Cu[:, :-1], neigh[:, :-1] & neigh[:, 1:]),
        (idx[:-1, :], idx[1:, :], Cv[:-1, :], neigh[:-1, :] & neigh[1:, :])):
    a, b, c = a2[face], b2[face], coef[face]
    for src, dst in ((a, b), (b, a)):
        m = src >= 0
        np.add.at(diag, src[m], -c[m])       # this cell owns the face either way
        mm = m & (dst >= 0)                  # off-diagonal only if the neighbour is an unknown
        rows.append(src[mm]); cols.append(dst[mm]); vals.append(c[mm])

L = sparse.csr_matrix(
    (np.concatenate(vals + [diag]),
     (np.concatenate(rows + [np.arange(n_unk)]), np.concatenate(cols + [np.arange(n_unk)]))),
    shape=(n_unk, n_unk))

ncomp, comp = connected_components(L, directed=False)
rowsum = np.asarray(L.sum(axis=1)).ravel()
dmax   = np.abs(L.diagonal()).max()
# a basin with no open face has zero row sums -> pure Neumann -> singular
singular = [bool(np.abs(rowsum[comp == c]).max() < 1e-9 * dmax) for c in range(ncomp)]
print(f"{n_unk} unknowns, {ncomp} basin(s): " +
      ", ".join(f"{int((comp==c).sum())} ({'singular' if singular[c] else 'well-posed'})"
                for c in range(ncomp)))

373675 unknowns, 1 basin(s): 373675 (well-posed)


In [5]:
# Factorise each basin once; reuse for every timestep.
blocks = []
for c in range(ncomp):
    sel = np.where(comp == c)[0]
    sub = L[sel][:, sel].tolil()
    if singular[c]:                       # pin one point to kill the constant null space
        sub[0, :] = 0; sub[0, 0] = 1.0
    blocks.append((sel, splu(sub.tocsc()), singular[c]))

def solve_phi(D2d):
    '''D2d: (ny,nx) column-integrated divergence in m/s -> phi on the T-grid.'''
    rhs = (D2d * area)[unknown]
    phi = np.zeros(n_unk)
    for sel, lu, sing in blocks:
        b = rhs[sel].copy()
        if sing:                          # enforce compatibility, then match the pin
            b -= b.mean(); b[0] = 0.0
        phi[sel] = lu.solve(b)
    out = np.zeros((ny, nx)); out[unknown] = phi
    return out
print("factorised")

factorised


## 3. Solve, one day at a time

In [6]:
def _open_uv(path, var):
    ds = xr.open_dataset(path, chunks={"time_counter": 1})
    da = ds[var].rename({"deptht": "z"})
    da = da.drop_vars([c for c in ("nav_lon", "nav_lat", "deptht") if c in da.coords])
    return da.assign_coords(x=np.arange(nx), y=np.arange(ny), z=np.arange(nz)).fillna(0.0)

U = _open_uv(U_PATH, "vozocrtx")
V = _open_uv(V_PATH, "vomecrty")
nt = U.sizes["time_counter"]

def col_divergence(u3, v3):
    '''(nz,ny,nx) -> (ny,nx) column-integrated divergence, m/s. Rim rows/cols are not closed.'''
    Fu, Fv = u3 * e2u * e3u, v3 * e1v * e3v
    d = np.zeros_like(Fu)
    d[:, :, 1:]  = Fu[:, :, 1:] - Fu[:, :, :-1]
    d[:, 1:, :] += Fv[:, 1:, :] - Fv[:, :-1, :]
    return d.sum(0) / area

du = np.zeros((nt, ny, nx)); dv = np.zeros((nt, ny, nx))
D0 = np.zeros((nt, ny, nx))

face_u = ocean[:, :-1] & ocean[:, 1:]
face_v = ocean[:-1, :] & ocean[1:, :]

for t in range(nt):
    u3 = U.isel(time_counter=t).values.astype("f8")
    v3 = V.isel(time_counter=t).values.astype("f8")
    D  = col_divergence(u3, v3)
    D0[t] = D
    phi = solve_phi(D)
    du[t, :, :-1] = np.where(face_u, -(phi[:, 1:] - phi[:, :-1]) / e1u[:, :-1], 0.0)
    dv[t, :-1, :] = np.where(face_v, -(phi[1:, :] - phi[:-1, :]) / e2v[:-1, :], 0.0)
    print(f"  day {t+1:2d}/{nt}   rms D {np.sqrt((D[unknown]**2).mean()):.2e}   "
          f"rms du {np.sqrt((du[t][unknown]**2).mean()):.2e} m/s")

/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'vozocrtx' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'vomecrty' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


  day  1/31   rms D 2.03e-07   rms du 1.22e-03 m/s
  day  2/31   rms D 2.20e-07   rms du 1.02e-03 m/s
  day  3/31   rms D 2.22e-07   rms du 7.42e-04 m/s
  day  4/31   rms D 2.19e-07   rms du 1.27e-03 m/s
  day  5/31   rms D 2.36e-07   rms du 1.29e-03 m/s
  day  6/31   rms D 1.90e-07   rms du 8.02e-04 m/s
  day  7/31   rms D 2.10e-07   rms du 1.24e-03 m/s
  day  8/31   rms D 2.42e-07   rms du 1.04e-03 m/s
  day  9/31   rms D 2.39e-07   rms du 1.11e-03 m/s
  day 10/31   rms D 2.57e-07   rms du 1.02e-03 m/s
  day 11/31   rms D 2.38e-07   rms du 1.31e-03 m/s
  day 12/31   rms D 2.41e-07   rms du 1.04e-03 m/s
  day 13/31   rms D 1.93e-07   rms du 1.21e-03 m/s
  day 14/31   rms D 2.09e-07   rms du 1.10e-03 m/s
  day 15/31   rms D 1.93e-07   rms du 8.45e-04 m/s
  day 16/31   rms D 2.19e-07   rms du 1.38e-03 m/s
  day 17/31   rms D 1.89e-07   rms du 9.21e-04 m/s
  day 18/31   rms D 2.56e-07   rms du 1.08e-03 m/s
  day 19/31   rms D 1.73e-07   rms du 7.45e-04 m/s
  day 20/31   rms D 2.21e-07   

## 4. Corrected U, V and the new W

In [7]:
dims = ("time_counter", "y", "x")
crd  = dict(time_counter=U.time_counter, y=np.arange(ny), x=np.arange(nx))
dU = xr.DataArray(du, dims=dims, coords=crd)
dV = xr.DataArray(dv, dims=dims, coords=crd)

_zc   = dict(z=np.arange(nz), y=np.arange(ny), x=np.arange(nx))
wet_u = xr.DataArray(e3u > 0, dims=("z", "y", "x"), coords=_zc)
wet_v = xr.DataArray(e3v > 0, dims=("z", "y", "x"), coords=_zc)

# --- linear weight  g = 2(z+H)/H : 0 at the seabed, 2 at the surface, column mean 1 ---
#
# Built on the u-face and v-face columns, NOT on the T-column. What has to be preserved is the
# flux through each face, sum_k g*e3u = Hu. Normalising g on e3t/H instead leaves that sum off
# by a median 4e-4 and up to 100% at some faces, and W(0) does not come back to zero.
def linear_weight(e3f, Hf):
    mid = np.cumsum(e3f, axis=0) - 0.5 * e3f            # depth of each cell centre, positive down
    Hs  = np.where(Hf > 0, Hf, 1.0)
    g   = np.where(e3f > 0, 2.0 * (Hf - mid) / Hs, 0.0)
    s   = (g * e3f).sum(axis=0)                          # force sum_k g*e3f == Hf exactly
    return np.where(s > 0, g * Hf / np.where(s > 0, s, 1.0), 0.0)

gU = xr.DataArray(linear_weight(e3u, Hu), dims=("z", "y", "x"), coords=_zc)
gV = xr.DataArray(linear_weight(e3v, Hv), dims=("z", "y", "x"), coords=_zc)

Uc = ((U + dU * gU) * wet_u).chunk({"time_counter": 1, "z": -1})
Vc = ((V + dV * gV) * wet_v).chunk({"time_counter": 1, "z": -1})

_e3u = xr.DataArray(e3u, dims=("z", "y", "x"), coords=wet_u.coords)
_e3v = xr.DataArray(e3v, dims=("z", "y", "x"), coords=wet_v.coords)
_e2u = xr.DataArray(e2u, dims=("y", "x"), coords=dict(y=np.arange(ny), x=np.arange(nx)))
_e1v = xr.DataArray(e1v, dims=("y", "x"), coords=_e2u.coords)
_area = xr.DataArray(area, dims=("y", "x"), coords=_e2u.coords)
_k    = xr.DataArray(np.arange(nz), dims="z", coords=dict(z=np.arange(nz)))
_mb   = xr.DataArray(mbathy, dims=("y", "x"), coords=_e2u.coords)

Fu = Uc * _e2u * _e3u
Fv = Vc * _e1v * _e3v
inc = -((Fu - Fu.shift(x=1)) + (Fv - Fv.shift(y=1))) / _area
Wc = inc.isel(z=slice(None, None, -1)).cumsum("z").isel(z=slice(None, None, -1))
Wc = Wc.assign_coords(z=inc.z).where(_k <= _mb, 0.0).fillna(0.0)
Wc = Wc.where((Wc.x > 0) & (Wc.y > 0) & (Wc.x < nx - 1) & (Wc.y < ny - 1), 0.0)
Wc

<xarray.DataArray (time_counter: 31, z: 50, y: 499, x: 1260)> Size: 8GB
dask.array<where, shape=(31, 50, 499, 1260), dtype=float64, chunksize=(1, 50, 250, 630), chunktype=numpy.ndarray>
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 248B 1993-01-01T12:00:00 ... ...
  * z             (z) int64 400B 0 1 2 3 4 5 6 7 8 ... 42 43 44 45 46 47 48 49
  * y             (y) int64 4kB 0 1 2 3 4 5 6 7 ... 492 493 494 495 496 497 498
  * x             (x) int64 10kB 0 1 2 3 4 5 6 ... 1254 1255 1256 1257 1258 1259
Attributes:
    units:               m s-1
    valid_min:           -10.0
    valid_max:           10.0
    online_operation:    N/A
    interval_operation:  86400
    interval_write:      86400
    associate:           time_counter deptht nav_lat nav_lon
    _ChunkSizes:         [  1   5 340 481]

## 5. Diagnostics

In [8]:
w0 = Wc.isel(z=0).values
print(f"1. surface residual BEFORE   rms {np.sqrt((D0[:, unknown]**2).mean()):.2e} m/s")
print(f"2. surface residual AFTER    rms {np.sqrt((w0[:, unknown]**2).mean()):.2e}  "
      f"max {np.abs(w0[:, unknown]).max():.2e} m/s")

rU = float(np.sqrt((U.isel(time_counter=0).values**2).mean()))
print(f"3. correction magnitude      rms du {np.sqrt((du[:, unknown]**2).mean()):.2e}  "
      f"rms dv {np.sqrt((dv[:, unknown]**2).mean()):.2e}   (rms U {rU:.2e})")
_a = np.abs(du[:, unknown]).ravel()
print(f"   |du| p50 {np.percentile(_a,50):.1e}  p90 {np.percentile(_a,90):.1e}  "
      f"p99 {np.percentile(_a,99):.1e}  max {_a.max():.1e} m/s")
print("   by water depth:")
for lo, hi, lab in [(0,50,"H < 50 m   "), (50,200,"50-200 m   "),
                    (200,1000,"200-1000 m "), (1000,1e9,"H > 1000 m ")]:
    s = unknown & (H >= lo) & (H < hi)
    if s.sum(): print(f"     {lab} rms du {np.sqrt((du[:, s]**2).mean()):.2e} m/s   ({s.sum()} columns)")

_t = min(1, nt - 1)
wG = (xr.open_dataset(W_PATH, chunks={"time_counter": 1}).vovecrtz
        .rename({"depthw": "z"}).assign_coords(x=np.arange(nx), y=np.arange(ny), z=np.arange(nz))
        .isel(time_counter=_t).values)
wc = Wc.isel(time_counter=_t).values
sl = (slice(1, None), slice(120, 320), slice(400, 700))     # interior, below the surface
g  = np.isfinite(wG[sl]) & np.isfinite(wc[sl])
print(f"4. corr(Wc, GLORYS W) interior   {np.corrcoef(wc[sl][g], wG[sl][g])[0,1]:.4f}")
print(f"5. NaN/Inf in Uc,Vc,Wc: "
      f"{bool(np.isnan(du).any() or np.isnan(dv).any() or not np.isfinite(w0).all())}")
# --- the weight itself ---
_err = np.abs((gU.values * e3u).sum(0) - Hu)[Hu > 0] / Hu[Hu > 0]
print(f"6. weight check  max |sum(g*e3u) - Hu| / Hu = {_err.max():.2e}")
_j, _i = 250, 500
_m = mbathy[_j, _i]
print(f"   g at (y={_j}, x={_i}), H={H[_j,_i]:.0f} m:  surface {gU.values[0,_j,_i]:.3f}"
      f"   mid {gU.values[_m//2,_j,_i]:.3f}   deepest cell {gU.values[_m-1,_j,_i]:.3f}")
print("   effective |du| by depth band (rms over the domain):")
_dm = np.cumsum(e3u, axis=0) - 0.5 * e3u
for _lo, _hi in [(0, 20), (100, 500), (500, 2000), (2000, 1e9)]:
    _s = (_dm >= _lo) & (_dm < _hi) & (e3u > 0) & unknown
    if _s.sum():
        _e = (np.abs(du)[:, None, :, :] * gU.values[None])[:, _s]
        print(f"     {_lo:>5.0f}-{_hi if _hi < 1e8 else 9999:>5.0f} m  rms {np.sqrt((_e**2).mean()):.2e} m/s")


1. surface residual BEFORE   rms 2.25e-07 m/s
2. surface residual AFTER    rms 2.37e-18  max 1.13e-16 m/s
3. correction magnitude      rms du 1.16e-03  rms dv 1.09e-03   (rms U 1.14e-01)
   |du| p50 6.6e-06  p90 7.6e-05  p99 1.7e-03  max 8.1e-02 m/s
   by water depth:
     H < 50 m    rms du 5.20e-03 m/s   (18157 columns)
     50-200 m    rms du 1.23e-03 m/s   (8567 columns)
     200-1000 m  rms du 2.36e-04 m/s   (10731 columns)
     H > 1000 m  rms du 3.86e-05 m/s   (336220 columns)


/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'vovecrtz' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


4. corr(Wc, GLORYS W) interior   1.0000
5. NaN/Inf in Uc,Vc,Wc: False
6. weight check  max |sum(g*e3u) - Hu| / Hu = 8.91e-16
   g at (y=250, x=500), H=4753 m:  surface 2.000   mid 1.934   deepest cell 0.029
   effective |du| by depth band (rms over the domain):
         0-   20 m  rms 1.18e-03 m/s
       100-  500 m  rms 7.16e-05 m/s
       500- 2000 m  rms 4.22e-05 m/s
      2000- 9999 m  rms 1.23e-05 m/s


## 6. Write

In [9]:
src = xr.open_dataset(U_PATH, chunks={})
xs, ys = src.x.values, src.y.values
nav_lon, nav_lat = src.nav_lon.values, src.nav_lat.values

def write(da, name, vdim, depth, path):
    out = (da.rename({"z": vdim}).rename(name)
             .transpose("time_counter", vdim, "y", "x")
             .assign_coords({vdim: depth.astype("float32"), "x": xs, "y": ys})
             .astype("float32").to_dataset())
    out = out.assign_coords(nav_lon=(("y", "x"), nav_lon), nav_lat=(("y", "x"), nav_lat))
    out[name].attrs = {"units": "m/s", "coordinates": "nav_lon nav_lat"}
    out.to_netcdf(path, encoding={name: dict(dtype="float32", zlib=COMPRESS,
                                             complevel=1 if COMPRESS else 0)})
    print("written", path)

write(Uc, "vozocrtx", "deptht", gdept_0, U_OUT)
write(Vc, "vomecrty", "deptht", gdept_0, V_OUT)
write(Wc, "vovecrtz", "depthw", gdepw_0, W_OUT)

/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'vozocrtx' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


written U_1993-01b.nc
written V_1993-01b.nc
written W_1993-01b.nc


---

## What to check

`W(0)` at machine zero, interior W still matching GLORYS, and diagnostic 6 confirming
$\sum_k g\,e_{3u} = H_u$ at every face.

The linear weight does not make the correction smaller — it moves it. The column-integrated
transport correction is unchanged by construction, so the same volume still has to be pushed
sideways; the weighting just concentrates it in the upper water column and clears it out of
the 100-500 m range. On the shelf, where the water column is only tens of metres deep, "upper
water column" is most of the column, so the shelf correction is not reduced much — compare
diagnostic 3 against the uniform run before choosing.
